In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl

In [2]:
data = pd.read_csv("dataset/customer_shopping_data.csv")
data["revenue"] = data["quantity"] * data["price"]
data["invoice_date"] = pd.to_datetime(data["invoice_date"],format='mixed')

In [3]:
data.head(3)

,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall,revenue
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,2022-05-08,Kanyon,7502.00
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,2021-12-12,Forum Istanbul,5401.53
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,2021-09-11,Metrocity,300.08


# Customers

##### Customer gender

In [5]:
gender = data.groupby(["gender"],as_index=False).agg(
    count_of_gender = ("gender","count")
)

gender

,gender,count_of_gender
0,Female,59482
1,Male,39975


##### Customer age

In [7]:
data["age_group"] = data["age"].map(lambda x: "[0-14] -> Children" if x<15 
                                    else "[15-24] -> Youth" if x<26 
                                    else "[25-64] -> Adults" if x<65 
                                    else "[65-99] -> Older Adults")

age_group = data.groupby(["age_group"],as_index=False).agg(
    age_groups = ("age_group","count")
).sort_values(by="age_groups",ascending=False).reset_index(drop=True)

age_group

,age_group,age_groups
0,[25-64] -> Adults,74671
1,[15-24] -> Youth,15359
2,[65-99] -> Older Adults,9427


In [8]:
data.columns

Index(['invoice_no', 'customer_id', 'gender', 'age', 'category', 'quantity',
       'price', 'payment_method', 'invoice_date', 'shopping_mall', 'revenue',
       'age_group'],
      dtype='object')

##### Top 10 customers based on sales performance

In [9]:
snapshot_date = pd.Timestamp("2024-01-01")

rfm_table = data.groupby(["customer_id"],as_index=False).agg(
    Recency = ("invoice_date",lambda x: (snapshot_date - x.max()).days),
    Frequency = ("quantity","sum"),
    Monetary = ("revenue","sum")
).sort_values(by="Recency",ascending=False)

rfm_table["R"] = pd.qcut(rfm_table["Recency"].rank(method="first",ascending=True),5,labels=[5,4,3,2,1]).astype(int)
rfm_table["F"] = pd.qcut(rfm_table["Frequency"].rank(method="first",ascending=True),5,labels=[1,2,3,4,5]).astype(int)
rfm_table["M"] = pd.qcut(rfm_table["Monetary"].rank(method="first",ascending=True),5,labels=[1,2,3,4,5]).astype(int)

rfm_table["Weighted_RFM_Score"] = ( rfm_table["R"] * 0.40 + rfm_table["F"] * 0.30 + rfm_table["M"] * 0.30 ).round(2)

rfm_table["Segment"] = pd.cut( rfm_table["Weighted_RFM_Score"], bins=[0, 2, 3, 4, 5],
    labels=[
        "At Risk/Lost",
        "Potential Customers",
        "Loyal Customers",
        "Champions"
    ],
    include_lowest=True
)

rfm_table.head(3)

,customer_id,Recency,Frequency,Monetary,R,F,M,Weighted_RFM_Score,Segment
78178,C421044,1095,3,365.94,1,2,3,1.9,At Risk/Lost
78781,C437854,1095,2,60.60,1,1,1,1.0,At Risk/Lost
4026,C112914,1095,4,83.68,1,4,1,1.9,At Risk/Lost


##### Customers loyalty over the past years
<p>Minimum of date: 2021-01-01</p>
<p>Maximum of date: 2023-12-02</p>

In [18]:
df = data.copy()

df["invoice_year"] = df["invoice_date"].dt.strftime("%Y")

loyalty = df.groupby(["customer_id","invoice_year"]).agg(
    customer_exists = ("invoice_year","nunique")
).sort_values(by="customer_id",ascending=False)

loyalty.head(3)

,,customer_exists
customer_id,invoice_year,
C999995,2021,1
C999976,2022,1
C999974,2022,1


In [20]:
df_stores = data.copy()

stores = df_stores.groupby(["customer_id","shopping_mall","payment_method"]).agg(
    total_revenue = ("revenue","sum")
).sort_values(by="customer_id",ascending=False)

stores

,,,total_revenue
customer_id,shopping_mall,payment_method,
C999995,Kanyon,Cash,1200.32
C999976,Metrocity,Debit Card,322.56
C999974,Forum Istanbul,Cash,7502.00
C999910,Mall of Istanbul,Debit Card,136.35
C999886,Kanyon,Debit Card,47.07
...,...,...,...
C100019,Metrocity,Credit Card,35.84
C100012,Kanyon,Cash,130.75
C100006,Cevahir AVM,Credit Card,322.56
